# 02 · Parameters, NULLs and exact types

Bind user values without SQL interpolation and preserve integers, decimals and missing observations.

Open this notebook in Google Colab with a CPU runtime, or use a Linux x86-64
Jupyter environment with Python 3.11 or newer. No database account is required.
Run the cells in order. The first cell downloads a verified Kelvo preview binary
and installs pinned Python readers; it does not compile Kelvo.

Every analytical SQL query below runs through the **Kelvo executable**. PyArrow
reads its Arrow output, and pandas/matplotlib format small results. The pinned
DuckDB driver materializes execution before delivering Arrow batches; delivery
backpressure is not a worker memory bound.


**Data:** [Palmer Penguins](https://allisonhorst.github.io/palmerpenguins/),
[CC0 license](https://github.com/allisonhorst/palmerpenguins/blob/8957207b78d6ccd1b4654a9dd9c9041b657478ab/DESCRIPTION).
Citation: Horst, Hill and Gorman (2020), *palmerpenguins: Palmer Archipelago
(Antarctica) penguin data*, [DOI: 10.5281/zenodo.3960218](https://doi.org/10.5281/zenodo.3960218).
The helper downloads the pinned original CSV and verifies its SHA-256.


[All notebooks](https://github.com/SYNEHQ/kelvo-go/blob/cargo/notebooks/README.md)
| [Kelvo documentation](https://github.com/SYNEHQ/kelvo-go/blob/cargo/docs/README.md)

## Set up a private lab


In [ ]:
# Default setup installs pinned Python packages and verifies the reviewed helper.
# Operators validating a checkout can explicitly set KELVO_NOTEBOOK_SKIP_INSTALL=1
# and KELVO_NOTEBOOK_HELPER=/absolute/path/to/kelvo_notebooks.py before starting.
import hashlib
import importlib.util
import os
from pathlib import Path
import shutil
import subprocess
import sys
import tempfile
import urllib.parse
import urllib.request

KELVO_NOTEBOOK_REF = "eacb7e2aedd8020575ecfede53db819112c40199"
KELVO_HELPER_SHA256 = "86696a66a3c903eb8f18d529fcef055e8bee72234da324b8776529138436d9a9"
if os.environ.get("KELVO_NOTEBOOK_SKIP_INSTALL") != "1":
    subprocess.run([
        sys.executable, "-m", "pip", "install", "--quiet",
        "pyarrow==25.0.1", "pandas==2.3.3", "matplotlib==3.10.7", "PyYAML==6.0.3",
    ], check=True)

helper_directory = None
local_helper = os.environ.get("KELVO_NOTEBOOK_HELPER")
if local_helper:
    helper_path = Path(local_helper).expanduser().resolve(strict=True)
else:
    helper_url = (
        "https://raw.githubusercontent.com/SYNEHQ/kelvo-go/"
        + KELVO_NOTEBOOK_REF + "/notebooks/kelvo_notebooks.py"
    )
    request = urllib.request.Request(helper_url, headers={"User-Agent": "kelvo-notebooks"})
    with urllib.request.urlopen(request, timeout=60) as response:
        final_url = response.geturl()
        if urllib.parse.urlsplit(final_url).scheme != "https" or final_url != helper_url:
            raise ValueError("Helper download did not retain its pinned HTTPS URL")
        payload = response.read(1024 * 1024 + 1)
    if len(payload) > 1024 * 1024:
        raise ValueError("Unexpected helper size")
    if hashlib.sha256(payload).hexdigest() != KELVO_HELPER_SHA256:
        raise ValueError("Helper checksum mismatch")
    helper_directory = Path(tempfile.mkdtemp(prefix="kelvo-notebook-helper-"))
    helper_path = helper_directory / "kelvo_notebooks.py"
    helper_path.write_bytes(payload)

spec = importlib.util.spec_from_file_location("kelvo_notebooks", helper_path)
helper = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = helper
spec.loader.exec_module(helper)
from kelvo_notebooks import Lab, remote_query

lab = Lab.create()
print("Kelvo workspace:", lab.root)


In [ ]:
import csv
from decimal import Decimal
penguins = lab.penguins()
config = lab.catalog(sources=[{"id": "penguins", "type": "csv", "path": str(penguins)}])


## Bind a filter

Parameter types are explicit. Use placeholders for values; source identifiers
come from the trusted catalog. A parameter cannot add SQL syntax.


In [ ]:
sql = "SELECT COUNT(*)::BIGINT AS observations FROM penguins WHERE species = ?"
selected = lab.query(config, sql, sources=["penguins"], parameters=[
    {"type": "string", "value": "Adelie"}
]).table()
literal = lab.query(config, sql, sources=["penguins"], parameters=[
    {"type": "string", "value": "Adelie' OR 1=1 --"}
]).table()
assert selected["observations"][0].as_py() == 152
assert literal["observations"][0].as_py() == 0
display(selected.to_pandas())


## Keep exact values exact

Send large integers as decimal strings. Send a decimal as a string and cast in
SQL; converting through a Python float would lose the intended representation.


In [ ]:
exact = lab.query(config, """
    SELECT CAST(? AS BIGINT) AS event_id,
           CAST(? AS DECIMAL(18,2)) AS amount,
           CAST(? AS VARCHAR) AS missing_note
""", sources=["penguins"], parameters=[
    {"type": "int64", "value": "9007199254740993"},
    {"type": "string", "value": "1234.56"},
    {"type": "null", "value": None},
]).table()
assert exact["event_id"][0].as_py() == 9007199254740993
assert exact["amount"][0].as_py() == Decimal("1234.56")
assert exact["missing_note"][0].as_py() is None
print(exact.schema)
display(exact.to_pandas())


## Interpret source missing values explicitly

The published CSV uses the literal `NA`. Normalize it before calculating a mean,
rather than silently treating missing mass as zero.


In [ ]:
missing = lab.query(config, """
    WITH typed AS (
        SELECT TRY_CAST(NULLIF(body_mass_g::VARCHAR, 'NA') AS DOUBLE) AS mass_g
        FROM penguins
    )
    SELECT COUNT(*)::BIGINT AS total,
           COUNT(mass_g)::BIGINT AS measured,
           COUNT(*) FILTER (WHERE mass_g IS NULL)::BIGINT AS missing,
           AVG(mass_g) AS mean_mass_g
    FROM typed
""", sources=["penguins"]).table().to_pylist()[0]
with penguins.open(newline="") as stream:
    rows = list(csv.DictReader(stream))
expected_missing = sum(row["body_mass_g"] in ("NA", "") for row in rows)
assert missing["total"] == len(rows)
assert missing["missing"] == expected_missing == 2
assert missing["measured"] + missing["missing"] == missing["total"]
assert missing["mean_mass_g"] > 0
print(missing)


## Clean up

Run this cell after inspecting or saving your results. Re-run the setup and data
cells if you want to restart the lesson after cleanup.


In [ ]:
# Close local processes and delete this lesson's private temporary workspace.
# Downloaded release/data caches, if configured by the helper, remain reusable.
lab.close()
if helper_directory is not None:
    shutil.rmtree(helper_directory, ignore_errors=True)
